In [0]:
df = spark.read.csv("/databricks-datasets/wine-quality/winequality-red.csv", header=True)
display(df)

データ読み込み & 型変換

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType, IntegerType

df = spark.read.csv(
    "/databricks-datasets/wine-quality/winequality-red.csv",
    header=True, sep=";"
)

# 数値型に変換（CSVはデフォルトで全部string）
numeric_cols = [c for c in df.columns if c != "quality"]
for col in numeric_cols:
    df = df.withColumn(col, F.col(col).cast(DoubleType()))
df = df.withColumn("quality", F.col("quality").cast(IntegerType()))

# ランクラベルを追加（3-5:低品質, 6:中品質, 7-8:高品質）
df = df.withColumn("quality_label",
    F.when(F.col("quality") <= 5, "low")
     .when(F.col("quality") == 6, "medium")
     .otherwise("high")
)

display(df)

品質別の統計サマリー

In [0]:
# 品質グレード別の各成分の平均を比較
summary = df.groupBy("quality_label").agg(
    F.count("*").alias("count"),
    F.round(F.avg("alcohol"), 2).alias("avg_alcohol"),
    F.round(F.avg("volatile acidity"), 3).alias("avg_volatile_acidity"),
    F.round(F.avg("sulphates"), 3).alias("avg_sulphates"),
    F.round(F.avg("citric acid"), 3).alias("avg_citric_acid"),
    F.round(F.avg("pH"), 3).alias("avg_pH"),
    F.round(F.avg("residual sugar"), 2).alias("avg_residual_sugar"),
)

display(summary.orderBy("avg_alcohol"))
# → display後に「可視化」タブで棒グラフにすると一目瞭然！

Databricks visualization. Run in Databricks to view.

In [0]:
import pandas as pd

# PandasへのブリッジでPythonの相関計算を活用
pdf = df.toPandas()

correlations = pdf.corr(numeric_only=True)["quality"].drop("quality").sort_values(ascending=False)
print("=== 品質(quality)との相関係数 ===")
print(correlations.to_string())

In [0]:
import mlflow
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import RandomForestClassifier
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

label_indexer = StringIndexer(inputCol="quality_label", outputCol="label")
feature_cols = [c for c in df.columns if c not in ["quality", "quality_label"]]
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features")
rf = RandomForestClassifier(numTrees=100, maxDepth=5, seed=42)
pipeline = Pipeline(stages=[label_indexer, assembler, rf])

train, test = df.randomSplit([0.8, 0.2], seed=42)

with mlflow.start_run(run_name="wine_quality_rf"):
    model = pipeline.fit(train)
    predictions = model.transform(test)
    
    evaluator = MulticlassClassificationEvaluator(metricName="accuracy")
    accuracy = evaluator.evaluate(predictions)
    
    # 手動でパラメータとメトリクスを記録
    mlflow.log_params({
        "numTrees": 100,
        "maxDepth": 5,
    })
    mlflow.log_metric("accuracy", accuracy)
    print(f"テスト精度: {accuracy:.4f}")

display(predictions.select("quality_label", "label", "prediction", "probability"))

Serverlessクラスターを使っているので mlflow.spark.autolog() が使えないことに注意する。

In [0]:
import pandas as pd
import matplotlib.pyplot as plt

# plt.rcParams['font.family'] = 'IPAexGothic'
# 日本語フォントを設定
# Serverlessクラスターだと apt-get の権限がないらしい

# モデルからFeature Importanceを抽出
rf_model = model.stages[-1]
importances = rf_model.featureImportances.toArray()

fi_df = pd.DataFrame({
    "feature": feature_cols,
    "importance": importances
}).sort_values("importance", ascending=True)

fig, ax = plt.subplots(figsize=(8, 5))
ax.barh(fi_df["feature"], fi_df["importance"], color="steelblue")
ax.set_xlabel("Feature Importance")
ax.set_title(" Wine Quality Prediction - Feature Importance")
plt.tight_layout()
display(fig)
plt.close()  # 自動描画を抑制